# 200-pair matching-rule ablation; builds the 2,819-image reference (`tab:rules`)

Renders and scores the 8 matching rules (random, mean, pixel, position, gaussian, nn, hard, sharp) on the first 200 pairs, and builds the canonical 2,819-image stratified reference (`ref_list.json`, `ref_inc.npy`, `ref_dino.npy`) every other table's KID depends on.

**Needs:** DTD, the CNT checkpoint.

**Maintained runnable path:** `experiments/rules_200.py` re-renders this table; `score_only.py` rescores it from cached features (`tab:rules-200`). Kept for provenance -- this is also where the canonical reference itself is defined.

# Matching-rule ablation — does the paper need OT correspondence at all?
Same pairs, same encoder/decoder/layout, same η; only the rule that assigns each source cell a target location changes.

| rule | target location of source cell $x$ | marginal constraint | appearance-aware |
|---|---|---|---|
| `ot` (default ε=0.05) | barycentric projection under entropic plan | yes | yes (diffuse) |
| `ot_sharp` (ε=0.005) | same, sharp plan | yes | yes |
| `exact` (Hungarian, 1024-pt) | assigned partner | yes (hard) | yes |
| `nn` | nearest target cell in feature space | **no** | yes |
| `gauss` | Monge map between fitted Gaussians | moments only | global only |
| `uniform` | target mean | yes (trivially) | **no** |
| `random` | random permutation partner | yes | **no** |
| `pixel_linear`, `texton_linear` | positional partner | — | **no** |

Outputs (all to Drive, resumable): per-pair CSV of KID-features, realism (Inception, DINO; 5 manifold draws), DINO placement, endpoint gap; paired Wilcoxon + bootstrap CIs vs `ot`; figures. Then two small ablations: feature preprocessing, subsample size $M$.

In [ ]:
# ============================================================
# 0. SETUP — identical pipeline to tm_scale1000 (CELL 4), plus a plan-returning Sinkhorn and per-pair seeding.
# ============================================================
!pip install -q pot plenoptic einops timm lpips
import os, sys, glob, random, time, gc, math, json
import numpy as np, pandas as pd, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageFile
from scipy.stats import wilcoxon
from google.colab import drive
drive.mount('/content/drive')
ImageFile.LOAD_TRUNCATED_IMAGES = True
device = 'cuda' if torch.cuda.is_available() else 'cpu'

BASE = os.environ.get('DATA_ROOT', '/content/data')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
OUT  = f'{BASE}/matching_ablation'; os.makedirs(OUT, exist_ok=True)
FIGS = f'{OUT}/figs'; os.makedirs(FIGS, exist_ok=True)
SCALE = f'{BASE}/tm_scale1000'          # saved n=1000 midpoints + manifest
TM50  = f'{BASE}/tm_comparison'         # saved n=50 run
TMP = '/content/tmp_imgs'; os.makedirs(TMP, exist_ok=True)
NUM = {}                                # macro name -> value, dumped at the end
def note(k, v): NUM[k] = v; print(f'  {k} = {v}')

LOCAL_DTD = '/content/dtd_images'
if len(glob.glob(f'{LOCAL_DTD}/*/*.jpg')) < 5000:
    os.system(f'cp -r "{BASE}/dtd/images" {LOCAL_DTD}')
DTD_ROOT = LOCAL_DTD
cats = sorted([c for c in os.listdir(DTD_ROOT) if os.path.isdir(f'{DTD_ROOT}/{c}')])
def dtd_path(c,s): return random.Random(s).choice(sorted(glob.glob(f'{DTD_ROOT}/{c}/*.jpg')))
print(f'{len(cats)} categories')

os.chdir('/content'); os.system('rm -rf cnt-siga24')
os.system('git clone -q https://github.com/phtu-cs/compositional-neural-textures.git cnt-siga24')
os.system('pip install -q -r cnt-siga24/requirements.in')
os.system('pip install -q "git+https://github.com/facebookresearch/detectron2.git"')
os.system(f'rm -rf /content/cnt-siga24/ckpts; ln -sfn "{BASE}/cnt_ckpts" /content/cnt-siga24/ckpts')
CNT_DIR='/content/cnt-siga24'; os.chdir(CNT_DIR); sys.path.insert(0,CNT_DIR)
from src.inference.core import TexAutoEncoderInference, load_image
model = TexAutoEncoderInference(ckpt_path=Path(f"{CNT_DIR}/ckpts/finetuned-model256.ckpt"), device=device)
IMG_SIZE = 256
FMS = [round(IMG_SIZE*r) for r in model.generator.feature_map_size_ratios]
EPS, ITERS, NSUB = 0.05, 100, 4096

_tc=[0]
def cnt_load(x, size=IMG_SIZE):
    if isinstance(x,(str,Path)): return load_image(Path(x), size).to(device)
    _tc[0]+=1; fp=f'{TMP}/t{_tc[0]%64}.png'; x.convert('RGB').save(fp)
    return load_image(Path(fp), size).to(device)
def to_pil(dec):
    a=((dec.squeeze(0).detach().cpu().clamp(-1,1)+1)/2).permute(1,2,0).numpy()
    return Image.fromarray((a*255).astype(np.uint8))
def get_gen_inputs(img):
    with torch.no_grad():
        blob=model.encode(img); L=model.splat(blob)
        return blob, L["layer_level_layouts"][-1][0]["blob_features"].clone(), L["layer_level_layouts"][-1][0]["score_img"].clone()
def decode_interp(blob_ref, bf, si):
    with torch.no_grad():
        L=model.splat(blob_ref)
        L["layer_level_layouts"][-1][0]["blob_features"]=bf; L["layer_level_layouts"][-1][0]["score_img"]=si
        return model.decode(L)
def build_style_maps(bf,si,fms):
    sp={}; s=si; sp[s.shape[-1]]=s
    while s.shape[-1]>min(fms):
        s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
    return {size: torch.einsum("bnc,bnhw->bchw",bf,sp[size]) for size in set(fms)}
def decode_with_style_maps(blob_ref, sm):
    with torch.no_grad():
        L=model.splat(blob_ref); si=L["layer_level_layouts"][-1][0]["score_img"]
        fms=[round(si.shape[-1]*r) for r in model.generator.feature_map_size_ratios]
        sp={}; s=si; sp[s.shape[-1]]=s
        while s.shape[-1]>min(fms):
            s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
        fi=sp[fms[0]][:,1:].amax(dim=1)[:,None].repeat(1,model.generator.n_embedding,1,1)
        x=fi.clone()
        for i in range(len(fms)):
            x=model.generator.spade_blocks[i](x,sm[fms[i]])
            if i==len(fms)-1: break
            elif fms[i]!=fms[i+1]: x=F.interpolate(x,size=x.shape[-1]*2,mode="bilinear",align_corners=False)
        return torch.tanh(x.unsqueeze(1)).flatten(1,2)

# --- Sinkhorn that returns the plan (needed for entropy, rho*, variance) ---
def sinkhorn_plan(src,tgt,eps=EPS,n_iters=ITERS):
    N,M=src.shape[0],tgt.shape[0]; C=torch.cdist(src,tgt).pow(2); K=torch.exp(-C/eps)
    a=torch.ones(N,1,device=src.device)/N; b=torch.ones(M,1,device=src.device)/M; u=torch.ones(N,1,device=src.device)
    for _ in range(n_iters): v=b/(K.T@u+1e-10); u=a/(K@v+1e-10)
    P=u*K*v.T; return P, C
def sinkhorn_transport(src,tgt,eps=EPS,n_iters=ITERS):
    P,_=sinkhorn_plan(src,tgt,eps,n_iters); return (P/(P.sum(1,keepdim=True)+1e-10))@tgt

def build_d(ia, ib):
    ba,bfa,sia=get_gen_inputs(ia); bb,bfb,sib=get_gen_inputs(ib)
    return {"blob_a":ba,"bf_a":bfa,"si_a":sia,"blob_b":bb,"bf_b":bfb,"si_b":sib}
def _grids(d):
    sa=build_style_maps(d["bf_a"],d["si_a"],FMS); sb=build_style_maps(d["bf_b"],d["si_b"],FMS)
    L=max(FMS); A,B=sa[L][0],sb[L][0]; C,H,W=A.shape
    return L,C,H,W, A.permute(1,2,0).reshape(-1,C), B.permute(1,2,0).reshape(-1,C)
_TAG={'s':1,'t':2,'a':3,'b':4}
def _sub(N, seed, tag):
    g=torch.Generator(device=device).manual_seed(int(seed)*7919+_TAG[tag])
    return torch.randperm(N,generator=g,device=device)[:NSUB]
def _dec(d, it, L, H, W, C, layout='a'):
    it=it.reshape(H,W,C).permute(2,0,1).unsqueeze(0)
    return decode_with_style_maps(d["blob_"+layout], {s:(it if s==L else F.interpolate(it,size=s,mode="bilinear",align_corners=False)) for s in set(FMS)})

def interp_ot(d, eta, seed=0, eps=EPS, return_plan=False):
    L,C,H,W,sf,tf=_grids(d); N=sf.shape[0]
    i_s=_sub(N,seed,'s'); i_t=_sub(N,seed,'t')
    with torch.no_grad(): P,Cm=sinkhorn_plan(sf[i_s],tf[i_t],eps); tt=(P/(P.sum(1,keepdim=True)+1e-10))@tf[i_t]
    nn=torch.cdist(sf,sf[i_s]).argmin(1)
    out=_dec(d,(1-eta)*sf+eta*tt[nn],L,H,W,C)
    return (out,dict(P=P,C=Cm,src=sf[i_s],tgt=tf[i_t],tt=tt,nn=nn,sf=sf,tf=tf)) if return_plan else out
def interp_ot_sym(d, eta, seed=0):
    # AS IMPLEMENTED: the A-grid and B-grid interpolants are averaged ELEMENTWISE AT THE SAME GRID POSITION
    # and decoded through A's layout. Write this honestly in the Method.
    L,C,H,W,af,bf=_grids(d); N=af.shape[0]
    ia=_sub(N,seed,'a'); ib=_sub(N,seed,'b')
    with torch.no_grad(): tab=sinkhorn_transport(af[ia],bf[ib]); tba=sinkhorn_transport(bf[ib],af[ia])
    na=torch.cdist(af,af[ia]).argmin(1); nb=torch.cdist(bf,bf[ib]).argmin(1)
    return _dec(d,0.5*(((1-eta)*af+eta*tab[na])+(eta*bf+(1-eta)*tba[nb])),L,H,W,C)
def interp_lin_pixel(d, eta, seed=0):
    sa=build_style_maps(d["bf_a"],d["si_a"],FMS); sb=build_style_maps(d["bf_b"],d["si_b"],FMS)
    return decode_with_style_maps(d["blob_a"], {s:(1-eta)*sa[s]+eta*sb[s] for s in set(FMS)})
def interp_lin_texton(d, eta, seed=0):
    return decode_interp(d["blob_a"], (1-eta)*d["bf_a"]+eta*d["bf_b"], (1-eta)*d["si_a"]+eta*d["si_b"])
FNS = {'pixel_linear':interp_lin_pixel,'texton_linear':interp_lin_texton,'ot':interp_ot,'ot_sym':interp_ot_sym}
def make_recon(X_pil):
    e=cnt_load(X_pil); return FNS['pixel_linear'](build_d(e,e),0.0)

# --- observers ---
import timm, lpips
dino = timm.create_model('vit_small_patch14_dinov2.lvd142m', pretrained=True, num_classes=0).to(device).eval()
for p in dino.parameters(): p.requires_grad_(False)
_M=torch.tensor([0.485,0.456,0.406],device=device).view(1,3,1,1); _S=torch.tensor([0.229,0.224,0.225],device=device).view(1,3,1,1)
@torch.no_grad()
def dino_feats(pils, bs=16):
    out=[]
    for i in range(0,len(pils),bs):
        xs=[torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/255. for p in pils[i:i+bs]]
        b=torch.stack(xs).to(device); b=F.interpolate(b,size=518,mode='bilinear',align_corners=False); b=(b-_M)/_S
        out.append(dino(b).cpu().numpy())
    return np.concatenate(out)
from torchvision.models import inception_v3, Inception_V3_Weights
_inc = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1, aux_logits=True).to(device).eval(); _inc.fc=torch.nn.Identity()
for p in _inc.parameters(): p.requires_grad_(False)
@torch.no_grad()
def inc_feats(pils, bs=24):
    out=[]
    for i in range(0,len(pils),bs):
        xs=[torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/255. for p in pils[i:i+bs]]
        b=torch.stack(xs).to(device); b=F.interpolate(b,size=299,mode='bilinear',align_corners=False); b=(b-_M)/_S
        out.append(_inc(b).cpu().numpy())
    return np.concatenate(out)
lp = lpips.LPIPS(net='alex').to(device).eval()
@torch.no_grad()
def lpips_d(p1,p2):
    t=lambda p: (torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/127.5-1)[None].to(device)
    return float(lp(t(p1),t(p2)))
print('ready')

In [ ]:
# ============================================================
# 1. MATCHING RULES — every rule returns a target location for each of the M sampled source cells, then shares
#    the SAME nn-extension, blend, and decode as interp_ot.
# ============================================================
from scipy.optimize import linear_sum_assignment

def _targets(sf, tf, rule, seed, M=NSUB):
    """sf,tf: full clouds (N,d). Returns (i_s, tt): sampled source indices and their target locations (M,d)."""
    N=sf.shape[0]; i_s=_sub(N,seed,'s'); i_t=_sub(N,seed,'t'); src, tgt = sf[i_s], tf[i_t]
    if rule=='ot':
        P,_=sinkhorn_plan(src,tgt,EPS);   tt=(P/(P.sum(1,keepdim=True)+1e-10))@tgt
    elif rule=='ot_sharp':
        P,_=sinkhorn_plan(src,tgt,0.005); tt=(P/(P.sum(1,keepdim=True)+1e-10))@tgt
    elif rule=='exact':
        n=1024; g=torch.Generator(device=device).manual_seed(seed+11)
        js=torch.randperm(M,generator=g,device=device)[:n]; is_=torch.randperm(M,generator=g,device=device)[:n]
        Cm=torch.cdist(src[is_],tgt[js]).pow(2).cpu().numpy(); r,c=linear_sum_assignment(Cm)
        tt=torch.empty_like(src); tt[:]=float('nan')
        tt[is_[torch.as_tensor(r,device=device)]]=tgt[js[torch.as_tensor(c,device=device)]]
        # unassigned sampled cells take the target of their nearest assigned neighbour (same nn rule as the full grid)
        ok=~torch.isnan(tt[:,0]); nn=torch.cdist(src[~ok],src[ok]).argmin(1); tt[~ok]=tt[ok][nn]
    elif rule=='nn':
        tt=tgt[torch.cdist(src,tgt).argmin(1)]
    elif rule=='gauss':
        mx,my=src.mean(0),tgt.mean(0); Xc,Yc=src-mx,tgt-my
        Sx=Xc.T@Xc/M+1e-4*torch.eye(src.shape[1],device=device); Sy=Yc.T@Yc/M+1e-4*torch.eye(src.shape[1],device=device)
        def msqrt(S): e,v=torch.linalg.eigh(S); return (v*e.clamp_min(0).sqrt())@v.T
        def misqrt(S): e,v=torch.linalg.eigh(S); return (v*e.clamp_min(1e-8).rsqrt())@v.T
        Sxh=msqrt(Sx); A=misqrt(Sx)@msqrt(Sxh@Sy@Sxh)@misqrt(Sx)
        tt=my+Xc@A.T
    elif rule=='uniform':
        tt=tgt.mean(0,keepdim=True).expand_as(src).clone()
    elif rule=='random':
        g=torch.Generator(device=device).manual_seed(seed+7); tt=tgt[torch.randperm(M,generator=g,device=device)]
    else: raise ValueError(rule)
    return i_s, tt

def interp_rule(d, eta, rule, seed=0):
    if rule=='pixel_linear': return interp_lin_pixel(d,eta)
    if rule=='texton_linear': return interp_lin_texton(d,eta)
    L,C,H,W,sf,tf=_grids(d); i_s,tt=_targets(sf,tf,rule,seed)
    nn=torch.cdist(sf,sf[i_s]).argmin(1)
    return _dec(d,(1-eta)*sf+eta*tt[nn],L,H,W,C)

RULES=['ot','ot_sharp','exact','nn','gauss','uniform','random','pixel_linear','texton_linear']
# quick visual sanity on one pair
A=Image.open(dtd_path('cracked',3)).convert('RGB'); B=Image.open(dtd_path('honeycombed',3)).convert('RGB'); d=build_d(cnt_load(A),cnt_load(B))
fig,ax=plt.subplots(1,len(RULES)+2,figsize=(2.2*(len(RULES)+2),2.4))
ax[0].imshow(A.resize((256,256))); ax[0].set_title('A')
for a,r in zip(ax[1:-1],RULES): a.imshow(to_pil(interp_rule(d,0.5,r,seed=0))); a.set_title(r,fontsize=8)
ax[-1].imshow(B.resize((256,256))); ax[-1].set_title('B')
for a in ax: a.axis('off')
plt.tight_layout(); plt.savefig(f'{FIGS}/rules_one_pair.png',dpi=150,bbox_inches='tight'); plt.show()

In [ ]:
# ============================================================
# 2. OBSERVERS — manifold realism (Kynkäänniemi et al.), KID, DINO placement, endpoint gap
# ============================================================
REF_LIST=f'{OUT}/ref_list.json'
if os.path.exists(REF_LIST): ref_files=json.load(open(REF_LIST))
else:
    random.seed(2819); ref_files=[]
    for c in cats: ref_files+=random.sample(sorted(glob.glob(f'{DTD_ROOT}/{c}/*.jpg')),60)
    ref_files=ref_files[:2819]; json.dump(ref_files,open(REF_LIST,'w'))
def _load128(f): return Image.open(f).convert('RGB').resize((128,128))
if os.path.exists(f'{OUT}/ref_inc.npy'): ref_inc=np.load(f'{OUT}/ref_inc.npy'); ref_dino=np.load(f'{OUT}/ref_dino.npy')
else:
    ims=[_load128(f) for f in ref_files]; ref_inc=inc_feats(ims); ref_dino=dino_feats(ims)
    np.save(f'{OUT}/ref_inc.npy',ref_inc); np.save(f'{OUT}/ref_dino.npy',ref_dino)

def kid(X, Y, n_sub=None, n_rep=20, seed=0):
    rng=np.random.RandomState(seed); dd=X.shape[1]; m=min(n_sub or len(X),len(X),len(Y)); vals=[]
    for _ in range(n_rep):
        x=X[rng.choice(len(X),m,replace=False)]; y=Y[rng.choice(len(Y),m,replace=False)]
        k=lambda a,b:(a@b.T/dd+1)**3; kxx,kyy,kxy=k(x,x),k(y,y),k(x,y)
        vals.append((kxx.sum()-np.trace(kxx))/(m*(m-1))+(kyy.sum()-np.trace(kyy))/(m*(m-1))-2*kxy.mean())
    return float(np.mean(vals)), float(np.std(vals))

class Manifold:
    """realism score: max_r radius_r / ||x - r|| over reference points r, k=3 NN radii, largest half of radii pruned."""
    def __init__(self, R, k=3, n=1000, seed=0):
        rng=np.random.RandomState(seed); self.R=R[rng.choice(len(R),min(n,len(R)),replace=False)]
        D=np.linalg.norm(self.R[:,None]-self.R[None],axis=-1); D.sort(1); rad=D[:,k]
        keep=rad<=np.median(rad); self.R=self.R[keep]; self.rad=rad[keep]
    def __call__(self, X):
        D=np.linalg.norm(X[:,None]-self.R[None],axis=-1); return (self.rad[None]/(D+1e-9)).max(1)
MANI_INC=[Manifold(ref_inc,seed=s) for s in range(5)]; MANI_DINO=[Manifold(ref_dino,seed=s) for s in range(5)]
def realism(F, manis): return np.median(np.stack([m(F) for m in manis]),0)
print('observers ready; ref n =',len(ref_inc))

In [ ]:
# ============================================================
# 3. MAIN RUN — n pairs from the saved tm_scale1000 manifest (same pairs as Table 2), all rules, eta=0.5 (+ eta=1 for endpoint gap)
# ============================================================
N_PAIRS=200
MAN=json.load(open(f'{SCALE}/manifest.json'))['pairs'][:N_PAIRS]
CSV=f'{OUT}/rules_n{N_PAIRS}.csv'; FEAT=f'{OUT}/rules_feats.npz'
done=pd.read_csv(CSV) if os.path.exists(CSV) else pd.DataFrame(columns=['k','rule'])
have=set(zip(done.k.astype(int),done.rule)); rows=done.to_dict('records')
feats=dict(np.load(FEAT,allow_pickle=True)) if os.path.exists(FEAT) else {}
for p_ in MAN:
    k=int(p_['k'])
    if all((k,r) in have for r in RULES): continue
    A=Image.open(f'{SCALE}/A/{k:04d}.png').convert('RGB'); B=Image.open(f'{SCALE}/B/{k:04d}.png').convert('RGB')
    d=build_d(cnt_load(A),cnt_load(B)); rA=to_pil(make_recon(A)); rB=to_pil(make_recon(B))
    fe=dino_feats([rA.resize((128,128)),rB.resize((128,128))]); dAB=np.linalg.norm(fe[0]-fe[1])
    for r in RULES:
        if (k,r) in have: continue
        mid=to_pil(interp_rule(d,0.5,r,seed=k)).resize((128,128)); end=to_pil(interp_rule(d,1.0,r,seed=k)).resize((128,128))
        fi=inc_feats([mid])[0]; fd=dino_feats([mid,end])
        feats[f'{k}_{r}_inc']=fi; feats[f'{k}_{r}_dino']=fd[0]
        rows.append(dict(k=k,rule=r,
            real_inc=float(realism(fi[None],MANI_INC)[0]), real_dino=float(realism(fd[0][None],MANI_DINO)[0]),
            place=float(np.linalg.norm(fd[0]-fe[0])/(np.linalg.norm(fd[0]-fe[0])+np.linalg.norm(fd[0]-fe[1])+1e-9)),
            endpoint_gap=float(np.linalg.norm(fd[1]-fe[1])/dAB)))
        if r=='ot' and k<6:
            Image.fromarray(np.asarray(mid)).save(f'{FIGS}/pair{k}_ot.png')
    if k%10==0:
        pd.DataFrame(rows).to_csv(CSV,index=False); np.savez(FEAT,**feats); print(k)
df=pd.DataFrame(rows); df.to_csv(CSV,index=False); np.savez(FEAT,**feats)

# ---- set-level KID per rule (vs the fixed reference), n=200
K={r: kid(np.stack([feats[f'{k}_{r}_inc'] for k in df.k.unique()]), ref_inc) for r in RULES}
# ---- per-pair paired comparisons vs OT, with bootstrap CI on the median difference
def boot_ci(x, n=2000, seed=0):
    rng=np.random.RandomState(seed); m=[np.median(rng.choice(x,len(x))) for _ in range(n)]; return np.percentile(m,[2.5,97.5])
piv={m: df.pivot(index='k',columns='rule',values=m) for m in ['real_inc','real_dino','place','endpoint_gap']}
print(f"{'rule':14}{'KID':>16}{'realInc':>9}{'realDINO':>10}{'|place-.5|':>11}{'endgap':>8}   paired vs ot: dRealInc [95%CI] p | dRealDINO [95%CI] p")
for r in RULES:
    line=f"{r:14}{K[r][0]:8.4f}±{K[r][1]:.4f}{piv['real_inc'][r].median():9.3f}{piv['real_dino'][r].median():10.3f}{(piv['place'][r]-.5).abs().median():11.3f}{piv['endpoint_gap'][r].median():8.3f}"
    if r!='ot':
        for m in ['real_inc','real_dino']:
            dlt=(piv[m][r]-piv[m]['ot']).dropna().values; lo,hi=boot_ci(dlt); p=wilcoxon(dlt).pvalue if np.any(dlt) else 1
            line+=f"   {np.median(dlt):+.3f} [{lo:+.3f},{hi:+.3f}] p={p:.2g} |"
    print(line)
json.dump({r:K[r] for r in RULES},open(f'{OUT}/kid_by_rule.json','w'),indent=1)

In [ ]:
# ============================================================
# 4. FIGURES — (a) grid: 8 pairs x all rules at eta=0.5; (b) eta sweep for uniform / random / nn / ot on one pair
# ============================================================
SHOW=[r for r in RULES]; ks=[int(p_['k']) for p_ in MAN[:8]]
fig,ax=plt.subplots(len(ks),len(SHOW)+2,figsize=(1.9*(len(SHOW)+2),1.9*len(ks)))
for i,k in enumerate(ks):
    A=Image.open(f'{SCALE}/A/{k:04d}.png').convert('RGB'); B=Image.open(f'{SCALE}/B/{k:04d}.png').convert('RGB'); d=build_d(cnt_load(A),cnt_load(B))
    ax[i,0].imshow(A.resize((256,256))); ax[i,-1].imshow(B.resize((256,256)))
    for j,r in enumerate(SHOW): ax[i,j+1].imshow(to_pil(interp_rule(d,0.5,r,seed=k)))
    for a in ax[i]: a.axis('off')
for j,t in enumerate(['A']+SHOW+['B']): ax[0,j].set_title(t,fontsize=8)
plt.tight_layout(); plt.savefig(f'{FIGS}/rules_grid.png',dpi=150,bbox_inches='tight'); plt.show()

k=ks[1]; A=Image.open(f'{SCALE}/A/{k:04d}.png').convert('RGB'); B=Image.open(f'{SCALE}/B/{k:04d}.png').convert('RGB'); d=build_d(cnt_load(A),cnt_load(B))
ETAS=[i/6 for i in range(7)]; rows_=['uniform','random','nn','ot']
fig,ax=plt.subplots(len(rows_),7,figsize=(14,2.1*len(rows_)))
for i,r in enumerate(rows_):
    for j,e in enumerate(ETAS): ax[i,j].imshow(to_pil(interp_rule(d,e,r,seed=k))); ax[i,j].axis('off'); ax[i,j].set_title(f'η={e:.2f}' if i==0 else '',fontsize=8)
    ax[i,0].text(-15,128,r,rotation=90,va='center',ha='right')
plt.tight_layout(); plt.savefig(f'{FIGS}/rules_sweep.png',dpi=150,bbox_inches='tight'); plt.show()

In [ ]:
# ============================================================
# 5. FEATURE PREPROCESSING ABLATION — raw / centred / unit-norm cost, n=50, OT default eps
# ============================================================
def interp_ot_pre(d, eta, pre, seed=0):
    L,C,H,W,sf,tf=_grids(d); N=sf.shape[0]; i_s=_sub(N,seed,'s'); i_t=_sub(N,seed,'t')
    src,tgt=sf[i_s],tf[i_t]
    if pre=='centred': cs,ct=src-src.mean(0),tgt-tgt.mean(0)
    elif pre=='unit':  cs,ct=F.normalize(src,dim=1),F.normalize(tgt,dim=1)
    else: cs,ct=src,tgt
    eps=EPS*float(torch.cdist(cs,ct).pow(2).median())/0.028   # keep eps at the same multiple of median cost as the default
    P,_=sinkhorn_plan(cs,ct,eps); tt=(P/(P.sum(1,keepdim=True)+1e-10))@tgt      # transport locations in RAW space
    nn=torch.cdist(sf,sf[i_s]).argmin(1); return _dec(d,(1-eta)*sf+eta*tt[nn],L,H,W,C)
PRE=f'{OUT}/preproc_n50.csv'
if os.path.exists(PRE): dp=pd.read_csv(PRE)
else:
    rows=[]
    for p_ in MAN[:50]:
        k=int(p_['k']); A=Image.open(f'{SCALE}/A/{k:04d}.png').convert('RGB'); B=Image.open(f'{SCALE}/B/{k:04d}.png').convert('RGB'); d=build_d(cnt_load(A),cnt_load(B))
        for pre in ['raw','centred','unit']:
            m=to_pil(interp_ot_pre(d,0.5,pre,seed=k)).resize((128,128)); fi=inc_feats([m])[0]; fd=dino_feats([m])[0]
            rows.append(dict(k=k,pre=pre,real_inc=float(realism(fi[None],MANI_INC)[0]),real_dino=float(realism(fd[None],MANI_DINO)[0])))
    dp=pd.DataFrame(rows); dp.to_csv(PRE,index=False)
print(dp.groupby('pre')[['real_inc','real_dino']].median())

In [ ]:
# ============================================================
# 6. SUBSAMPLE SIZE M — quality/runtime curve, n=30 pairs
# ============================================================
def interp_ot_M(d, eta, M, seed=0):
    L,C,H,W,sf,tf=_grids(d); N=sf.shape[0]
    g=torch.Generator(device=device).manual_seed(seed*7919+1); i_s=torch.randperm(N,generator=g,device=device)[:M]
    g=torch.Generator(device=device).manual_seed(seed*7919+2); i_t=torch.randperm(N,generator=g,device=device)[:M]
    P,_=sinkhorn_plan(sf[i_s],tf[i_t],EPS); tt=(P/(P.sum(1,keepdim=True)+1e-10))@tf[i_t]
    nn=torch.cdist(sf,sf[i_s]).argmin(1); return _dec(d,(1-eta)*sf+eta*tt[nn],L,H,W,C)
MS=[512,1024,2048,4096,8192]; MC=f'{OUT}/subsample_curve.csv'
if os.path.exists(MC): dm=pd.read_csv(MC)
else:
    rows=[]
    for p_ in MAN[:30]:
        k=int(p_['k']); A=Image.open(f'{SCALE}/A/{k:04d}.png').convert('RGB'); B=Image.open(f'{SCALE}/B/{k:04d}.png').convert('RGB'); d=build_d(cnt_load(A),cnt_load(B))
        for M in MS:
            torch.cuda.synchronize(); t0=time.perf_counter(); out=interp_ot_M(d,0.5,M,seed=k); torch.cuda.synchronize(); t=time.perf_counter()-t0
            m=to_pil(out).resize((128,128)); fi=inc_feats([m])[0]; fd=dino_feats([m])[0]
            rows.append(dict(k=k,M=M,sec=t,real_inc=float(realism(fi[None],MANI_INC)[0]),real_dino=float(realism(fd[None],MANI_DINO)[0])))
    dm=pd.DataFrame(rows); dm.to_csv(MC,index=False)
g=dm.groupby('M')[['sec','real_inc','real_dino']].median(); print(g)
fig,ax=plt.subplots(1,2,figsize=(8,3)); ax[0].semilogx(g.index,g.sec,'o-'); ax[0].set_xlabel('M'); ax[0].set_ylabel('s / midpoint (interp+decode)')
ax[1].semilogx(g.index,g.real_inc,'o-',label='Inception'); ax[1].semilogx(g.index,g.real_dino,'s-',label='DINO'); ax[1].set_xlabel('M'); ax[1].set_ylabel('realism'); ax[1].legend()
plt.tight_layout(); plt.savefig(f'{FIGS}/subsample_curve.png',dpi=150,bbox_inches='tight'); plt.show()

In [ ]:
# ============================================================
# 3b. FID per matching rule, from the saved Inception features (n=200 vs the fixed reference)
# ============================================================
from scipy import linalg
def fid(X, Y):
    mx, my = X.mean(0), Y.mean(0); Sx, Sy = np.cov(X, rowvar=False), np.cov(Y, rowvar=False)
    covmean, _ = linalg.sqrtm(Sx @ Sy, disp=False)
    if np.iscomplexobj(covmean): covmean = covmean.real
    return float(((mx-my)**2).sum() + np.trace(Sx + Sy - 2*covmean))

feats = dict(np.load(FEAT, allow_pickle=True)); df = pd.read_csv(CSV); ks = sorted(df.k.unique())
Xr = {r: np.stack([feats[f'{k}_{r}_inc'] for k in ks]) for r in RULES}

# bootstrap over pairs so the ordering comes with a CI
rng = np.random.RandomState(0); B = 200; boot = {r: [] for r in RULES}
for _ in range(B):
    idx = rng.choice(len(ks), len(ks), replace=True)
    for r in RULES: boot[r].append(fid(Xr[r][idx], ref_inc))
FID = {r: (fid(Xr[r], ref_inc), np.percentile(boot[r], [2.5, 97.5])) for r in RULES}

# pairwise "rule beats ot" probability under the same bootstrap draws
draws = {r: np.array(boot[r]) for r in RULES}
print(f"{'rule':14}{'FID':>8}{'95% CI':>18}{'P(FID < ot)':>13}")
for r in sorted(RULES, key=lambda r: FID[r][0]):
    p = float((draws[r] < draws['ot']).mean()) if r != 'ot' else float('nan')
    print(f"{r:14}{FID[r][0]:8.1f}   [{FID[r][1][0]:6.1f},{FID[r][1][1]:6.1f}]{p:13.2f}")
json.dump({r: [FID[r][0], *FID[r][1]] for r in RULES}, open(f'{OUT}/fid_by_rule.json', 'w'), indent=1)

In [ ]:
# ============================================================
# 7. EXACT ASSIGNMENT AS THE OPERATOR — all 1000 pairs of the Table 2 manifest.
#    Saves midpoints to {SCALE}/exact/ (same layout as the other method dirs), scores them with the
#    same observers, and re-scores the SAVED ot / pixel_linear / texton_linear / mid(TM) PNGs with the
#    same observers so every comparison is paired and apples-to-apples.
# ============================================================
from scipy import linalg
MAN_ALL = json.load(open(f'{SCALE}/manifest.json'))['pairs']
EXD = f'{SCALE}/exact'; os.makedirs(EXD, exist_ok=True)
EXCSV = f'{OUT}/exact_n1000.csv'

# --- 7a. check that 1024 points is not limiting: exact at 1024 vs 2048 on 20 pairs, DINO distance between the two decodes
def interp_exact(d, eta, seed, n_exact=1024):
    L,C,H,W,sf,tf=_grids(d); N=sf.shape[0]; i_s=_sub(N,seed,'s'); i_t=_sub(N,seed,'t'); src,tgt=sf[i_s],tf[i_t]
    g=torch.Generator(device=device).manual_seed(seed+11)
    js=torch.randperm(NSUB,generator=g,device=device)[:n_exact]; is_=torch.randperm(NSUB,generator=g,device=device)[:n_exact]
    r,c=linear_sum_assignment(torch.cdist(src[is_],tgt[js]).pow(2).cpu().numpy())
    tt=torch.full_like(src,float('nan')); tt[is_[torch.as_tensor(r,device=device)]]=tgt[js[torch.as_tensor(c,device=device)]]
    ok=~torch.isnan(tt[:,0]); tt[~ok]=tt[ok][torch.cdist(src[~ok],src[ok]).argmin(1)]
    nn=torch.cdist(sf,sf[i_s]).argmin(1); return _dec(d,(1-eta)*sf+eta*tt[nn],L,H,W,C)

chk=[]
for p_ in MAN_ALL[:20]:
    k=int(p_['k']); A=Image.open(f'{SCALE}/A/{k:04d}.png').convert('RGB'); B=Image.open(f'{SCALE}/B/{k:04d}.png').convert('RGB'); d=build_d(cnt_load(A),cnt_load(B))
    t0=time.perf_counter(); m1=to_pil(interp_exact(d,0.5,k,1024)); t1=time.perf_counter()-t0
    t0=time.perf_counter(); m2=to_pil(interp_exact(d,0.5,k,2048)); t2=time.perf_counter()-t0
    f=dino_feats([m1,m2]); chk.append(dict(k=k,dino_1024_vs_2048=float(np.linalg.norm(f[0]-f[1])),lpips=lpips_d(m1,m2),sec1024=t1,sec2048=t2))
chk=pd.DataFrame(chk); print(chk.median(numeric_only=True)); note('ExactSubsampleCheckLPIPS',round(float(chk.lpips.median()),3)); note('ExactSec1024',round(float(chk.sec1024.median()),2))

# --- 7b. generate exact midpoints + eta=1 endpoints for all 1000 pairs (resumable)
for p_ in MAN_ALL:
    k=int(p_['k']); fp=f'{EXD}/{k:04d}.png'
    if os.path.exists(fp) and os.path.exists(f'{EXD}/{k:04d}_end.png'): continue
    A=Image.open(f'{SCALE}/A/{k:04d}.png').convert('RGB'); B=Image.open(f'{SCALE}/B/{k:04d}.png').convert('RGB'); d=build_d(cnt_load(A),cnt_load(B))
    to_pil(interp_exact(d,0.5,k)).save(fp); to_pil(interp_exact(d,1.0,k)).save(f'{EXD}/{k:04d}_end.png')
    if k%50==0: print('gen',k)

# --- 7c. score exact AND the saved baselines with the same observers, paired by k
METHODS={'exact':EXD,'ot':f'{SCALE}/ot','ot_sym':f'{SCALE}/ot_sym','pixel_linear':f'{SCALE}/pixel_linear','texton_linear':f'{SCALE}/texton_linear','tm':f'{SCALE}/mid'}
METHODS={m:p for m,p in METHODS.items() if os.path.isdir(p)}
if os.path.exists(EXCSV): sc=pd.read_csv(EXCSV)
else:
    rows=[]; FE={}
    for p_ in MAN_ALL:
        k=int(p_['k'])
        rA=to_pil(make_recon(Image.open(f'{SCALE}/A/{k:04d}.png').convert('RGB'))).resize((128,128))
        rB=to_pil(make_recon(Image.open(f'{SCALE}/B/{k:04d}.png').convert('RGB'))).resize((128,128))
        fe=dino_feats([rA,rB]); dAB=np.linalg.norm(fe[0]-fe[1])
        for m,pdir in METHODS.items():
            f=f'{pdir}/{k:04d}.png'
            if not os.path.exists(f): continue
            mid=Image.open(f).convert('RGB').resize((128,128)); fi=inc_feats([mid])[0]; fd=dino_feats([mid])[0]; FE[f'{k}_{m}']=fi
            row=dict(k=k,method=m,real_inc=float(realism(fi[None],MANI_INC)[0]),real_dino=float(realism(fd[None],MANI_DINO)[0]),
                     place=float(np.linalg.norm(fd-fe[0])/(np.linalg.norm(fd-fe[0])+np.linalg.norm(fd-fe[1])+1e-9)))
            if m=='exact':
                fend=dino_feats([Image.open(f'{EXD}/{k:04d}_end.png').convert('RGB').resize((128,128))])[0]; row['endpoint_gap']=float(np.linalg.norm(fend-fe[1])/dAB)
            rows.append(row)
        if k%100==0: pd.DataFrame(rows).to_csv(EXCSV,index=False); np.savez(f'{OUT}/exact_feats.npz',**FE); print('score',k)
    sc=pd.DataFrame(rows); sc.to_csv(EXCSV,index=False); np.savez(f'{OUT}/exact_feats.npz',**FE)

# --- 7d. set-level FID/KID at n=1000 (and n=50 descriptive), per method, vs the fixed reference
FE=dict(np.load(f'{OUT}/exact_feats.npz',allow_pickle=True))
def fid(X,Y):
    mx,my=X.mean(0),Y.mean(0); Sx,Sy=np.cov(X,rowvar=False),np.cov(Y,rowvar=False); cm,_=linalg.sqrtm(Sx@Sy,disp=False)
    return float(((mx-my)**2).sum()+np.trace(Sx+Sy-2*(cm.real if np.iscomplexobj(cm) else cm)))
piv={c: sc.pivot(index='k',columns='method',values=c) for c in ['real_inc','real_dino','place']}
print(f"\n{'method':14}{'FID1000':>9}{'KID1000':>16}{'realInc':>9}{'realDINO':>10}{'|place-.5|':>11}   paired vs exact: dRealInc [95%CI] p")
for m in METHODS:
    ks_m=[k for k in sc[sc.method==m].k]; X=np.stack([FE[f'{k}_{m}'] for k in ks_m])
    F=fid(X,ref_inc); Kd=kid(X,ref_inc,n_sub=1000)
    line=f"{m:14}{F:9.1f}{Kd[0]:9.4f}±{Kd[1]:.4f}{piv['real_inc'][m].median():9.3f}{piv['real_dino'][m].median():10.3f}{(piv['place'][m]-.5).abs().median():11.3f}"
    if m!='exact':
        dlt=(piv['real_inc']['exact']-piv['real_inc'][m]).dropna().values; lo,hi=boot_ci(dlt); p=wilcoxon(dlt).pvalue
        line+=f"   {np.median(dlt):+.3f} [{lo:+.3f},{hi:+.3f}] p={p:.2g}"
    print(line)
    note(f'FIDk_{m}',round(F,1)); note(f'KIDk_{m}',round(Kd[0],4)); note(f'RealInc_{m}',round(float(piv['real_inc'][m].median()),3)); note(f'RealDino_{m}',round(float(piv['real_dino'][m].median()),3)); note(f'Place_{m}',round(float(piv['place'][m].median()),3))
note('EndpointGap_exact',round(float(sc[sc.method=='exact'].endpoint_gap.median()),3))
json.dump(NUM,open(f'{OUT}/numbers_exact.json','w'),indent=1)